# Information Retrieval — Term 481 — Assignment 1
## Boolean Retrieval, Indexing & Preprocessing

**Based on:** Topic 1 (*Introduction to IR* + Boolean Retrieval) and Lecture 2 / Topic 2 (*Indexing & Preprocessing*).

### Instructions
- Work through each exercise in order. Each exercise has a **"Your Code"** cell for you to complete.


---
## Part 1 — Term–Document Incidence Matrix & Boolean Retrieval

We reuse the idea from the Lecture 1 notebook but with a **new** collection of 6 short English documents:


In [4]:
collection = {
    "d1": "the cat sat on the mat",
    "d2": "the dog sat on the log",
    "d3": "the cat chased the dog",
    "d4": "a mat is not a log",
    "d5": "the dog and the cat are friends",
    "d6": "cats and dogs sat together",
}
collection

{'d1': 'the cat sat on the mat',
 'd2': 'the dog sat on the log',
 'd3': 'the cat chased the dog',
 'd4': 'a mat is not a log',
 'd5': 'the dog and the cat are friends',
 'd6': 'cats and dogs sat together'}

### Exercise 1.1 — Build the Term–Document Incidence Matrix by hand (2 points)

Before writing any code, fill in the table below (replace the `?` with `1` if the term appears in the document, `0` otherwise). Use the **raw** words as they appear (no stemming yet) — do NOT split "cats"/"cat" or "dogs"/"dog".

| term | d1 | d2 | d3 | d4 | d5 | d6 |
|---|---|---|---|---|---|---|
| the | 1 | 1 | 1 | 0 | 1 | 0 |
| cat | 1 | 0 | 1 | 0 | 1 | 0 |
| sat | 1 | 1 | 0 | 0 | 0 | 1 |
| on | 1 | 1 | 0 | 0 | 0 | 0 |
| mat | 1 | 0 | 0 | 1 | 0 | 0 |
| dog | 0 | 1 | 1 | 0 | 1 | 0 |
| log | 0 | 1 | 0 | 1 | 0 | 0 |
| chased | 0 | 0 | 1 | 0 | 0 | 0 |
| a | 0 | 0 | 0 | 1 | 0 | 0 |
| is | 0 | 0 | 0 | 1 | 0 | 0 |
| not | 0 | 0 | 0 | 1 | 0 | 0 |
| and | 0 | 0 | 0 | 0 | 1 | 1 |
| are | 0 | 0 | 0 | 0 | 1 | 0 |
| friends | 0 | 0 | 0 | 0 | 1 | 0 |
| cats | 0 | 0 | 0 | 0 | 0 | 1 |
| dogs | 0 | 0 | 0 | 0 | 0 | 0 |
| together | 0 | 0 | 0 | 0 | 0 | 1 |


### Exercise 1.2 — Implement the matrix builder in code (5 points)

Fill in the three helper functions below (mirroring the ones used in the Lecture 1 notebook):
- `get_terms(collection)` → list of **all** term occurrences (with repetition) across the collection.
- `get_unique_terms(terms)` → list of terms with duplicates removed (keep first-seen order).
- `term_document_incidence_matrix(collection)` → dict mapping each unique term to its 0/1 incidence vector, one entry per document (in the collection's iteration order).


In [24]:
 def get_terms(collection):
        terms=[]
        for doc in collection:
            for term in collection[doc].split() :
                terms.append(term)
        return terms
  

 def get_unique_terms(terms):
        unique_terms=[]
        for d in terms :
            if d not in unique_terms:
               unique_terms.append(d)
        return unique_terms  

 def term_document_incidence_matrix(collection):
        docs_collection={}
        for doc in collection:
            if doc not in boolean_operators :
                docs_colllection[doc]=get_unique_terms(collection[doc].split())
        return docs_colllection

 
 def display_dict(dic):
        print("\n")
        for i in dic:
            print (i , " : " ,dic[i])
        print("\n")

In [3]:
# The output generated by the code written to answer this question should be as follows:

### Exercise 1.3 — Boolean Retrieval (5 points)

Implement `boolean_retrieval(query, collection)` supporting `AND`, `OR`, and `NOT`, evaluated **left to right** (no operator precedence needed), following the same design as the Lecture 1 notebook. Then answer the three queries below:

1. `cat AND dog`
2. `mat OR log`
3. `dog AND NOT cat`


In [25]:
def boolean_operator_processing(op, v1, v2=None):
    if op == 'AND':
        return [b1 & b2 for b1, b2 in zip(v1, v2)]
    elif op == 'OR':
        return [b1 | b2 for b1, b2 in zip(v1, v2)]
    elif op == 'NOT':
        return [1 - b for b in v1]

def boolean_retrieval(query, collection):
    td_matrix = term_document_incidence_matrix(collection)
    tokens = query.split()
    
    current_vector = None
    pending_op = None
    i = 0
    
    while i < len(tokens):
        token = tokens[i]
        
        if token in ('AND', 'OR'):
            pending_op = token
            i += 1
        elif token == 'NOT':
            i += 1
            term = tokens[i]
            vec = boolean_operator_processing('NOT', td_matrix[term])
            if current_vector is None:
                current_vector = vec
            else:
                current_vector = boolean_operator_processing(pending_op, current_vector, vec)
            i += 1
        else:
            vec = td_matrix[token]
            if current_vector is None:
                current_vector = vec
            else:
                current_vector = boolean_operator_processing(pending_op, current_vector, vec)
            i += 1
            
    return current_vector

In [6]:
# The output generated by the code written to answer this question should be as follows:

Query: 'cat AND dog'        -> {'d1': 0, 'd2': 0, 'd3': 1, 'd4': 0, 'd5': 1, 'd6': 0}  | matching docs: ['d3', 'd5']
,Query: 'mat OR log'         -> {'d1': 1, 'd2': 1, 'd3': 0, 'd4': 1, 'd5': 0, 'd6': 0}  | matching docs: ['d1', 'd2', 'd4']
,Query: 'dog AND NOT cat'    -> {'d1': 0, 'd2': 1, 'd3': 0, 'd4': 0, 'd5': 0, 'd6': 0}  | matching docs: ['d2']


---
## Part 2 — Inverted Index Construction & Query Processing

Recall the 3-step construction method from Topic 2 (Indexing & Preprocessing):

1. **Term sequence** — build a list of `(term, docID)` pairs.
2. **Sort** — sort the pairs by term, then by docID.
3. **Dictionary & Postings** — group by term into a *postings list* of docIDs (with document frequency `df`).

We'll build it for the two mini-documents:

> Doc 1: *"I did enact Julius Caesar I was killed i' the Capitol Brutus killed me"*
>
> Doc 2: *"So let it be with Caesar The noble Brutus hath told you Caesar was ambitious"*


In [7]:
docs = {
    1: "I did enact Julius Caesar I was killed i the Capitol Brutus killed me",
    2: "So let it be with Caesar The noble Brutus hath told you Caesar was ambitious",
}

### Exercise 2.1 — Build the inverted index (5 points)

Implement `build_inverted_index(docs)` that:
1. Lowercases every token,
2. Builds the sorted `(term, docID)` sequence,
3. Groups it into a dictionary: `term -> {"df": <#docs containing term>, "postings": [sorted list of docIDs, one entry per occurrence]}`.

(For simplicity, keep one posting per *occurrence*, i.e. `caesar` in doc 2 — which occurs twice — should appear twice in its postings list, matching the slide's "Core indexing step" table.)


In [27]:
def build_inverted_index(docs):

    pairs = []
    for doc_id, text in docs.items():
        for token in text.split():
            pairs.append((token.lower(), doc_id))
    
    pairs.sort(key=lambda x: (x[0], x[1]))
    
    
    inverted_index = {}
    for term, doc_id in pairs:
        if term not in inverted_index:
            inverted_index[term] = {
                "df": 0,
                "postings": []
            }
        inverted_index[term]["postings"].append(doc_id)
        

    for term in inverted_index:
        inverted_index[term]["df"] = len(set(inverted_index[term]["postings"]))
        
    return inverted_index

In [8]:
# The output generated by the code written to answer this question should be as follows:

ambitious  df=1  postings=[2]
,be         df=1  postings=[2]
,brutus     df=2  postings=[1, 2]
,caesar     df=2  postings=[1, 2, 2]
,capitol    df=1  postings=[1]
,did        df=1  postings=[1]
,enact      df=1  postings=[1]
,hath       df=1  postings=[2]
,i          df=1  postings=[1, 1, 1]
,it         df=1  postings=[2]
,julius     df=1  postings=[1]
,killed     df=1  postings=[1, 1]
,let        df=1  postings=[2]
,me         df=1  postings=[1]
,noble      df=1  postings=[2]
,so         df=1  postings=[2]
,the        df=2  postings=[1, 2]
,told       df=1  postings=[2]
,was        df=2  postings=[1, 2]
,with       df=1  postings=[2]
,you        df=1  postings=[2]


### Exercise 2.2 — AND merge algorithm (5 points)

Implement `and_merge(p1, p2)` that intersects **two sorted, duplicate-free** postings lists in `O(len(p1) + len(p2))` — i.e., use the two-pointer "merge" technique shown in the slides, **not** Python's `set` intersection (which would defeat the purpose of the exercise, even though it gives the same result).

Test it on:
```
wink:  [1, 2, 4, 8, 16, 32, 64]
drink: [1, 2, 3, 5, 8, 13, 21, 34]
```


In [28]:
def and_merge(p1, p2):
    answer = []
    i = 0
    j = 0
    
    while i < len(p1) and j < len(p2):
        if p1[i] == p2[j]:
            answer.append(p1[i])
            i += 1
            j += 1
        elif p1[i] < p2[j]:
            i += 1
        else:
            j += 1
            
    return answer

In [9]:
# The output generated by the code written to answer this question should be as follows:

wink AND drink -> [1, 2, 8]


### Exercise 2.3 — OR merge algorithm (5 points)

Now implement `or_merge(p1, p2)` (union of two sorted, duplicate-free lists), also with the two-pointer technique (`O(len(p1)+len(p2))`, no sets).


In [29]:
def or_merge(p1, p2):
    answer = []
    i = 0
    j = 0
    
    # دمج العناصر طالما توجد عناصر في القائمتين
    while i < len(p1) and j < len(p2):
        if p1[i] == p2[j]:
            answer.append(p1[i])
            i += 1
            j += 1
        elif p1[i] < p2[j]:
            answer.append(p1[i])
            i += 1
        else:
            answer.append(p2[j])
            j += 1
            
    # إضافة العناصر المتبقية من القائمة الأولى إن وجدت
    while i < len(p1):
        answer.append(p1[i])
        i += 1
        
    # إضافة العناصر المتبقية من القائمة الثانية إن وجدت
    while j < len(p2):
        answer.append(p2[j])
        j += 1
        
    return answer

In [10]:
# The output generated by the code written to answer this question should be as follows:

wink OR drink -> [1, 2, 3, 4, 5, 8, 13, 16, 21, 32, 34, 64]


### Exercise 2.4 — Answer with the real inverted index (5 points)

Using `inv_index` from Exercise 2.1 (and `set()` to drop duplicate docIDs from postings before merging), answer: **`caesar AND brutus`** and **`caesar OR ambitious`**.


In [32]:
docs = {
    1: "I did enact Julius Caesar I was killed i the Capitol Brutus killed me",
    2: "So let it be with Caesar The noble Brutus hath told you Caesar was ambitious",
}

def build_inverted_index(docs):
    pairs = []
    for doc_id, text in docs.items():
        for token in text.split():
            pairs.append((token.lower(), doc_id))
    pairs.sort(key=lambda x: (x[0], x[1]))
    
    inverted_index = {}
    for term, doc_id in pairs:
        if term not in inverted_index:
            inverted_index[term] = {"df": 0, "postings": []}
        inverted_index[term]["postings"].append(doc_id)
        
    for term in inverted_index:
        inverted_index[term]["df"] = len(set(inverted_index[term]["postings"]))
        
    return inverted_index

def and_merge(p1, p2):
    answer = []
    i, j = 0, 0
    while i < len(p1) and j < len(p2):
        if p1[i] == p2[j]:
            answer.append(p1[i])
            i += 1
            j += 1
        elif p1[i] < p2[j]:
            i += 1
        else:
            j += 1
    return answer

def or_merge(p1, p2):
    answer = []
    i, j = 0, 0
    while i < len(p1) and j < len(p2):
        if p1[i] == p2[j]:
            answer.append(p1[i])
            i += 1
            j += 1
        elif p1[i] < p2[j]:
            answer.append(p1[i])
            i += 1
        else:
            answer.append(p2[j])
            j += 1
    while i < len(p1):
        answer.append(p1[i])
        i += 1
    while j < len(p2):
        answer.append(p2[j])
        j += 1
    return answer

def not_merge(p1, all_doc_ids):
    return [doc_id for doc_id in all_doc_ids if doc_id not in p1]

def get_postings(term, inv_index):
    term = term.lower()
    if term in inv_index:
        return sorted(list(set(inv_index[term]["postings"])))
    return []

def boolean_retrieval_inv_index(query, inv_index, all_doc_ids=[1, 2]):
    tokens = query.split()
    current_postings = None
    pending_op = None
    i = 0
    
    while i < len(tokens):
        token = tokens[i]
        
        if token in ('AND', 'OR'):
            pending_op = token
            i += 1
        elif token == 'NOT':
            i += 1
            term = tokens[i]
            postings = not_merge(get_postings(term, inv_index), all_doc_ids)
            if current_postings is None:
                current_postings = postings
            else:
                current_postings = and_merge(current_postings, postings) if pending_op == 'AND' else or_merge(current_postings, postings)
            i += 1
        else:
            postings = get_postings(token, inv_index)
            if current_postings is None:
                current_postings = postings
            else:
                current_postings = and_merge(current_postings, postings) if pending_op == 'AND' else or_merge(current_postings, postings)
            i += 1
            
    return current_postings

In [12]:
# The output generated by the code written to answer this question should be as follows:

caesar AND brutus   -> [1, 2]
,caesar OR  ambitious-> [1, 2]


---
## Part 3 — Text Preprocessing Pipeline (Tokenization → Stopping → Stemming)

We will build the full pipeline shown in the Topic 2 slides:

> Input: *"This is an example sentence of how the pre-processing is applied to text in information retrieval. It includes: Tokenization, Stop Words Removal, and Stemming"*
>
> Expected output: `exampl sentenc pre process appli text inform retriev includ token stop word remov stem`

This exercise uses `nltk`'s Porter Stemmer (the classic algorithm covered in the slides).


In [15]:
import re
from nltk.stem import PorterStemmer

STOPWORDS = {
    "a", "an", "the", "is", "are", "was", "were", "this", "that", "of", "to",
    "in", "on", "for", "and", "it", "how", "with", "as", "by", "at", "from",
}

### Exercise 3.1 — Tokenization (5 points)

Implement `tokenize(text)` that:
- lower-cases the text,
- splits on any run of non-letter characters (so punctuation, digits, and hyphens act as separators — following the *"just split at non-letter characters"* common practice from the slides).


In [33]:
import re

def tokenize(text):
    text = text.lower()
    
    tokens = re.split(r'[^a-z]+', text)
    
    return [t for t in tokens if t]

In [16]:
# The output generated by the code written to answer this question should be as follows:

['this', 'is', 'an', 'example', 'sentence', 'of', 'how', 'the', 'pre', 'processing', 'is', 'applied', 'to', 'text', 'in', 'information', 'retrieval', 'it', 'includes', 'tokenization', 'stop', 'words', 'removal', 'and', 'stemming']


### Exercise 3.2 — Stop-word Removal (5 points)

Implement `remove_stopwords(tokens)` using the `STOPWORDS` set defined above.


In [34]:
def remove_stopwords(tokens):
    return [token for token in tokens if token not in STOPWORDS]

In [19]:
# The output generated by the code written to answer this question should be as follows:

['example', 'sentence', 'pre', 'processing', 'applied', 'text', 'information', 'retrieval', 'includes', 'tokenization', 'stop', 'words', 'removal', 'stemming']


### Exercise 3.3 — Stemming (5 points)

Implement `stem_tokens(tokens)` using `nltk.stem.PorterStemmer`.


In [35]:
from nltk.stem import PorterStemmer

def stem_tokens(tokens):
    stemmer = PorterStemmer()
    return [stemmer.stem(token) for token in tokens]

In [21]:
# The output generated by the code written to answer this question should be as follows:

['exampl', 'sentenc', 'pre', 'process', 'appli', 'text', 'inform', 'retriev', 'includ', 'token', 'stop', 'word', 'remov', 'stem']
,exampl sentenc pre process appli text inform retriev includ token stop word remov stem


### Exercise 3.4 — Full Pipeline (3 points)

Combine 3.1–3.3 into a single `preprocess(text)` function and verify it reproduces the golden output exactly.


In [36]:
import re
from nltk.stem import PorterStemmer

STOPWORDS = {
    "a", "an", "the", "is", "are", "was", "were", "this", "that", "of", "to",
    "in", "on", "for", "and", "it", "how", "with", "as", "by", "at", "from",
}

def preprocess(text):
    tokens = [t for t in re.split(r'[^a-z]+', text.lower()) if t]
    
    filtered_tokens = [token for token in tokens if token not in STOPWORDS]
    
    stemmer = PorterStemmer()
    stemmed_tokens = [stemmer.stem(token) for token in filtered_tokens]
    
    return " ".join(stemmed_tokens)

In [23]:
# The output generated by the code written to answer this question should be as follows:

Output:   exampl sentenc pre process appli text inform retriev includ token stop word remov stem
,Expected: exampl sentenc pre process appli text inform retriev includ token stop word remov stem


**Reflection:** if a different stop-word list or stemmer is used, minor differences are expected and normal — this is exactly why preprocessing choices (which stop-word list? which stemmer?) are design decisions that must be documented and applied *consistently* to both documents and queries (as emphasized in the Lecture 2 code notebook, where the *same* `normalize()`/`stem()` functions are re-applied to a search term before looking it up in the index).


---
## Part 4 — Positional Index & Phrase Queries

Recall from the slides: a **positional index** stores, for each term, *which positions* it occurs at within each document (not just *which documents*). This lets us answer **phrase queries** like `"to be"` by checking that two terms occur in the *same* document at *consecutive* positions.


In [25]:
phrase_docs = {
    1: "to be or not to be that is the question",
    2: "I want to be a good friend to be honest",
    3: "being or not being is not the same as to be",
}

### Exercise 4.1 — Build a Positional Index (5 points)

Implement `build_positional_index(docs)` returning `term -> {doc_id: [list of positions (0-based)]}`.


In [37]:
from collections import defaultdict

def build_positional_index(docs):
    positional_index = defaultdict(lambda: defaultdict(list))
    
    for doc_id, text in docs.items():
        tokens = text.lower().split()
        for position, token in enumerate(tokens):
            positional_index[token][doc_id].append(position)
            
    return {term: dict(doc_map) for term, doc_map in positional_index.items()}

In [26]:
# The output generated by the code written to answer this question should be as follows:

'to'  -> {1: [0, 4], 2: [2, 7], 3: [9]}
,'be'  -> {1: [1, 5], 2: [3, 8], 3: [10]}


**Expected:** `'to' -> {1: [0, 4], 2: [2, 7], 3: [9]}` and `'be' -> {1: [1, 5], 2: [3, 8], 3: [10]}`.


### Exercise 4.2 — Phrase Query (5 points)

Implement `phrase_query(pos_index, phrase)` for a **two-word** phrase `"w1 w2"`: it should return the list of docIDs where `w2` occurs at position `p+1` for some occurrence of `w1` at position `p` (i.e. the words are adjacent, in order). Test it on `"to be"`.


In [38]:
def phrase_query(pos_index, phrase):
    words = phrase.lower().split()
    w1, w2 = words[0], words[1]
    
    matching_docs = []
    
    if w1 not in pos_index or w2 not in pos_index:
        return matching_docs
        
    common_docs = set(pos_index[w1].keys()) & set(pos_index[w2].keys())
    
    for doc_id in sorted(common_docs):
        pos1_list = pos_index[w1][doc_id]
        pos2_set = set(pos_index[w2][doc_id])
        
        for p in pos1_list:
            if (p + 1) in pos2_set:
                matching_docs.append(doc_id)
                break  
                
    return matching_docs

In [39]:
# The output generated by the code written to answer this question should be as follows:

Note doc 3 ("... as to be") matches, while a pure Boolean `to AND be` query would also match, say, a hypothetical document containing "be nice to everyone" even though the phrase "to be" never occurs there — this is exactly why phrase queries need position information rather than just document-level term membership.
